In [1]:
# Set project paths.
from pathlib import Path
import os
import sys

def find_project_root():
    current = Path.cwd()

    for folder in [current] + list(current.parents):
        if (folder / "Data").exists() and (folder / "Notebooks").exists():
            return folder

    raise FileNotFoundError("Could not find project root. Make sure Data and Notebooks folders exist.")

project_folder = find_project_root()
notebook_folder = project_folder / "Notebooks"

os.chdir(project_folder)

print("Project folder:", project_folder)
print("Notebook folder:", notebook_folder)

Project folder: /Users/mac/Library/CloudStorage/OneDrive-UniversityofKeele/Dissertation/send-ev-project
Notebook folder: /Users/mac/Library/CloudStorage/OneDrive-UniversityofKeele/Dissertation/send-ev-project/Notebooks


In [2]:
# Import packages.
import pandas as pd
import os

In [3]:
# Set year and file path.
year = 2022

Solcast_Path = "Data/Solcast/Solcast_2022-2023_unformatted.csv"

if not os.path.exists(Solcast_Path):
    print("Solcast file not found")
else:
    print("Solcast file found")

Solcast file found


In [4]:
# Load input files.
df = pd.read_csv(Solcast_Path)

stats = pd.read_csv("Data/Analysis/Data_Tables/csv/statistics.csv")

consts = pd.read_csv("Data/Analysis/Const/Const.csv", index_col=0)

df.head()

,air_temp,clearsky_gti,cloud_opacity,ghi,gti,surface_pressure,snow_depth,wind_speed_100m,weather_type,period_end,period
0,12,0,96.9,0,0,996.2,0.0,9.5,OVERCAST,2022-01-01T00:05:00+00:00,PT5M
1,12,0,96.9,0,0,996.2,0.0,9.6,OVERCAST,2022-01-01T00:10:00+00:00,PT5M
2,12,0,96.8,0,0,996.2,0.0,9.7,OVERCAST,2022-01-01T00:15:00+00:00,PT5M
3,13,0,96.8,0,0,996.2,0.0,9.7,OVERCAST,2022-01-01T00:20:00+00:00,PT5M
4,13,0,96.8,0,0,996.2,0.0,9.8,OVERCAST,2022-01-01T00:25:00+00:00,PT5M


In [5]:
# Parse timestamps.
format_date = "%Y-%m-%dT%H:%M:%S%z"

df["DateTime"] = pd.to_datetime(
    df["period_end"],
    format=format_date,
)

df["DateTime"] = df["DateTime"].dt.tz_localize(None)

df.index = df["DateTime"]

df.head()

,air_temp,clearsky_gti,cloud_opacity,ghi,gti,surface_pressure,snow_depth,wind_speed_100m,weather_type,period_end,period,DateTime
DateTime,,,,,,,,,,,,
2022-01-01 00:05:00,12,0,96.9,0,0,996.2,0.0,9.5,OVERCAST,2022-01-01T00:05:00+00:00,PT5M,2022-01-01 00:05:00
2022-01-01 00:10:00,12,0,96.9,0,0,996.2,0.0,9.6,OVERCAST,2022-01-01T00:10:00+00:00,PT5M,2022-01-01 00:10:00
2022-01-01 00:15:00,12,0,96.8,0,0,996.2,0.0,9.7,OVERCAST,2022-01-01T00:15:00+00:00,PT5M,2022-01-01 00:15:00
2022-01-01 00:20:00,13,0,96.8,0,0,996.2,0.0,9.7,OVERCAST,2022-01-01T00:20:00+00:00,PT5M,2022-01-01 00:20:00
2022-01-01 00:25:00,13,0,96.8,0,0,996.2,0.0,9.8,OVERCAST,2022-01-01T00:25:00+00:00,PT5M,2022-01-01 00:25:00


In [6]:
# Filter selected year.
df = df.loc[
    (df.index >= f"{year}-01-01 00:05")
    & (df.index <= f"{year}-12-31 23:55")
]

df.head()

,air_temp,clearsky_gti,cloud_opacity,ghi,gti,surface_pressure,snow_depth,wind_speed_100m,weather_type,period_end,period,DateTime
DateTime,,,,,,,,,,,,
2022-01-01 00:05:00,12,0,96.9,0,0,996.2,0.0,9.5,OVERCAST,2022-01-01T00:05:00+00:00,PT5M,2022-01-01 00:05:00
2022-01-01 00:10:00,12,0,96.9,0,0,996.2,0.0,9.6,OVERCAST,2022-01-01T00:10:00+00:00,PT5M,2022-01-01 00:10:00
2022-01-01 00:15:00,12,0,96.8,0,0,996.2,0.0,9.7,OVERCAST,2022-01-01T00:15:00+00:00,PT5M,2022-01-01 00:15:00
2022-01-01 00:20:00,13,0,96.8,0,0,996.2,0.0,9.7,OVERCAST,2022-01-01T00:20:00+00:00,PT5M,2022-01-01 00:20:00
2022-01-01 00:25:00,13,0,96.8,0,0,996.2,0.0,9.8,OVERCAST,2022-01-01T00:25:00+00:00,PT5M,2022-01-01 00:25:00


In [7]:
# Calculate maximum solar.
df["P_MaxSolar"] = df["gti"] * (consts.loc["A_pvUnit"] * consts.loc["N_pv"]).values[0] / 1e3

df[["gti", "P_MaxSolar"]].head()

,gti,P_MaxSolar
DateTime,,
2022-01-01 00:05:00,0,0.0
2022-01-01 00:10:00,0,0.0
2022-01-01 00:15:00,0,0.0
2022-01-01 00:20:00,0,0.0
2022-01-01 00:25:00,0,0.0


In [8]:
# Calculate expected solar.
df["expected"] = stats.loc["eta_Solar", f"{year}"] * df["P_MaxSolar"]

df[["P_MaxSolar", "expected"]].head()

,P_MaxSolar,expected
DateTime,,
2022-01-01 00:05:00,0.0,0.0
2022-01-01 00:10:00,0.0,0.0
2022-01-01 00:15:00,0.0,0.0
2022-01-01 00:20:00,0.0,0.0
2022-01-01 00:25:00,0.0,0.0


In [9]:
# Save expected solar.
df.to_csv(
    f"Data/DEOP/{year}_Expected.csv",
    index=False,
    columns=["DateTime", "expected"],
)

In [10]:
# Save cleaned Solcast.
df.to_csv(
    f"Data/Solcast/Solcast_{year}.csv",
    index=False,
    columns=[
        "air_temp",
        "gti",
        "surface_pressure",
        "snow_depth",
        "cloud_opacity",
        "ghi",
        "clearsky_gti",
        "wind_speed_100m",
        "weather_type",
        "DateTime",
    ],
)

In [11]:
# Check saved files.
expected_check = pd.read_csv(f"Data/DEOP/{year}_Expected.csv")
solcast_check = pd.read_csv(f"Data/Solcast/Solcast_{year}.csv")

print(expected_check.head())
print(solcast_check.head())

              DateTime  expected
0  2022-01-01 00:05:00       0.0
1  2022-01-01 00:10:00       0.0
2  2022-01-01 00:15:00       0.0
3  2022-01-01 00:20:00       0.0
4  2022-01-01 00:25:00       0.0
   air_temp  gti  surface_pressure  snow_depth  cloud_opacity  ghi  \
0        12    0             996.2         0.0           96.9    0   
1        12    0             996.2         0.0           96.9    0   
2        12    0             996.2         0.0           96.8    0   
3        13    0             996.2         0.0           96.8    0   
4        13    0             996.2         0.0           96.8    0   

   clearsky_gti  wind_speed_100m weather_type             DateTime  
0             0              9.5     OVERCAST  2022-01-01 00:05:00  
1             0              9.6     OVERCAST  2022-01-01 00:10:00  
2             0              9.7     OVERCAST  2022-01-01 00:15:00  
3             0              9.7     OVERCAST  2022-01-01 00:20:00  
4             0              9.8    